# Numeric Inference: Predicting Video Popularity from Title Embeddings

This notebook explores the predictive power of video title embeddings for estimating video views. We train a hierarchical linear model using low-dimensional semantic representations.

## 1. Setup and Configuration

Install necessary libraries and mount Google Drive.

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy matplotlib seaborn statsmodels

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sentence_transformers import SentenceTransformer

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab or drive mount failed.')

# Configuration
CONFIG = {
    'INPUT_PATH': '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channels_structured.json',
    'OUTPUT_DIR': '/content/drive/MyDrive/numeric_inference_outputs',
    'EMBEDDING_MODEL_NAME': 'all-MiniLM-L6-v2',
    'GLOBAL_PCA_COMPONENTS': 15,
    'LOCAL_PCA_COMPONENTS': 5,
    'MIN_VIDEOS_PER_CHANNEL': 40,
    'REUSE_CACHE': True
}

if not os.path.exists(CONFIG['OUTPUT_DIR']):
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

embedding_model = SentenceTransformer(CONFIG['EMBEDDING_MODEL_NAME'])

## 2. Data Loading and Filtering

Load the structured channel data and filter for channels with at least 40 videos.

In [ ]:
def load_and_filter_data(file_path):
    if not os.path.exists(file_path):
        print(f'Error: File not found at {file_path}')
        return []

    with open(file_path, 'r') as f:
        data = json.load(f)

    filtered_data = [channel for channel in data if len(channel['videos']) >= CONFIG['MIN_VIDEOS_PER_CHANNEL']]

    print(f'Original channels: {len(data)}')
    print(f'Filtered channels (>= {CONFIG["MIN_VIDEOS_PER_CHANNEL"]} videos): {len(filtered_data)}')

    return filtered_data

channels = load_and_filter_data(CONFIG['INPUT_PATH'])

## 3. Embedding Video Titles

Embed all video titles using the specified model. We cache embeddings to Google Drive to avoid redundant computations.

In [ ]:
def get_embedding_cache_path():
    return os.path.join(CONFIG['OUTPUT_DIR'], 'video_title_embeddings_latest.json')

def load_embedding_cache():
    path = get_embedding_cache_path()
    if CONFIG['REUSE_CACHE'] and os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {}

def save_embedding_cache(cache):
    path = get_embedding_cache_path()
    with open(path, 'w') as f:
        json.dump(cache, f)

    # Versioned copy
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    versioned_path = os.path.join(CONFIG['OUTPUT_DIR'], f'video_title_embeddings_{timestamp}.json')
    with open(versioned_path, 'w') as f:
        json.dump(cache, f)

embedding_cache = load_embedding_cache()

def embed_titles(channels_data):
    all_titles = []
    for channel in channels_data:
        for video in channel['videos']:
            if video['title'] not in embedding_cache:
                all_titles.append(video['title'])

    if all_titles:
        print(f'Embedding {len(all_titles)} new titles...')
        new_embeddings = embedding_model.encode(all_titles, show_progress_bar=True)
        for title, emb in zip(all_titles, new_embeddings):
            embedding_cache[title] = emb.tolist()
        save_embedding_cache(embedding_cache)
    else:
        print('All titles already embedded and cached.')

if channels:
    embed_titles(channels)

## 4. Data Splitting

Split the data into 80% training and 20% testing per channel. Save both files in Google drive, and document both paths.

In [ ]:
train_data = []
test_data = []

for channel in channels:
    videos = channel['videos']
    train_vids, test_vids = train_test_split(videos, test_size=0.2, random_state=42)

    train_data.append({
        'channel_id': channel['channel_id'],
        'channel_name': channel['channel_name'],
        'videos': train_vids
    })

    test_data.append({
        'channel_id': channel['channel_id'],
        'channel_name': channel['channel_name'],
        'videos': test_vids
    })

# Save splits
train_path = os.path.join(CONFIG['OUTPUT_DIR'], 'train_structured_latest.json')
test_path = os.path.join(CONFIG['OUTPUT_DIR'], 'test_structured_latest.json')

with open(train_path, 'w') as f:
    json.dump(train_data, f, indent=2)
with open(test_path, 'w') as f:
    json.dump(test_data, f, indent=2)

# Versioned copies
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_path_train = os.path.join(CONFIG['OUTPUT_DIR'], f'train_structured_{timestamp}.json')
save_path_test = os.path.join(CONFIG['OUTPUT_DIR'], f'test_structured_{timestamp}.json')

with open(save_path_train, 'w') as f:
    json.dump(train_data, f, indent=2)
with open(save_path_test, 'w') as f:
    json.dump(test_data, f, indent=2)

print(f'Training split saved to: {train_path} (and {save_path_train})')
print(f'Testing split saved to: {test_path} (and {save_path_test})')

## 5. Dimensionality Reduction (PCA)

Reduce the embedding dimensionality to 15 using PCA, fitted on the entire training set.

In [ ]:
def get_all_embeddings(data_split):
    embeddings = []
    for channel in data_split:
        for video in channel['videos']:
            embeddings.append(embedding_cache[video['title']])
    return np.array(embeddings)

train_embeddings = get_all_embeddings(train_data)
test_embeddings = get_all_embeddings(test_data)

print(f'Train embeddings shape: {train_embeddings.shape}')
print(f'Test embeddings shape: {test_embeddings.shape}')

pca = PCA(n_components=CONFIG['GLOBAL_PCA_COMPONENTS'], random_state=42)
train_pca = pca.fit_transform(train_embeddings)
test_pca = pca.transform(test_embeddings)

print(f'Train PCA shape: {train_pca.shape}')
print(f'Test PCA shape: {test_pca.shape}')
print(f'Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}')

## 6. Model Training: Hierarchical/Mixed-Effects styled Approach

Normalize views across all channels and use global weights to select significant dimensions for channel-specific models.

In [ ]:
# 1. Prepare Global Training Data
all_train_log_views = []
for channel in train_data:
    views = [v['views'] for v in channel['videos']]
    all_train_log_views.extend(np.log1p(views))

all_train_log_views = np.array(all_train_log_views)

# 2. Global Normalization
GLOBAL_MEAN = all_train_log_views.mean()
GLOBAL_STD = all_train_log_views.std()

print(f'Global Mean (log views): {GLOBAL_MEAN:.4f}')
print(f'Global Std (log views): {GLOBAL_STD:.4f}')

norm_train_log_views = (all_train_log_views - GLOBAL_MEAN) / GLOBAL_STD

# 3. Train Global Model for Feature Selection
X_global = sm.add_constant(train_pca)
global_model = sm.OLS(norm_train_log_views, X_global).fit()

print('\nGlobal Model Summary (Statsmodels OLS):')
print(global_model.summary().tables[1])

# 4. Select Significant Dimensions (Top 5 by absolute coefficient value)
coeffs = global_model.params[1:] # Exclude const
abs_coeffs = np.abs(coeffs)
top_dim_indices = np.argsort(abs_coeffs)[-CONFIG['LOCAL_PCA_COMPONENTS']:]
selected_dims = sorted(top_dim_indices.tolist())

print(f'\nSelected top {CONFIG["LOCAL_PCA_COMPONENTS"]} dimensions: {selected_dims}')

# 5. Train Per-Channel Local Models
channel_models = {}
training_summaries = []
current_train_idx = 0

for channel in train_data:
    num_videos = len(channel['videos'])
    channel_train_pca = train_pca[current_train_idx : current_train_idx + num_videos, selected_dims]
    channel_norm_log_views = norm_train_log_views[current_train_idx : current_train_idx + num_videos]

    model = LinearRegression()
    model.fit(channel_train_pca, channel_norm_log_views)

    train_preds_norm = model.predict(channel_train_pca)
    train_r2 = r2_score(channel_norm_log_views, train_preds_norm)
    
    # Denormalize for original scale MAE
    train_preds_log = train_preds_norm * GLOBAL_STD + GLOBAL_MEAN
    channel_log_views = channel_norm_log_views * GLOBAL_STD + GLOBAL_MEAN
    train_mae_log = mean_absolute_error(channel_log_views, train_preds_log)

    channel_models[channel['channel_id']] = model

    training_summaries.append({
        'channel_name': channel['channel_name'],
        'num_videos': num_videos,
        'train_r2': train_r2,
        'train_mae_log': train_mae_log,
        'coeffs': model.coef_
    })

    current_train_idx += num_videos

df_train_summary = pd.DataFrame(training_summaries)
print('\nLocal Channel Training Summary (Top 10 by R2):')
display(df_train_summary.sort_values('train_r2', ascending=False).head(10))

print(f'Mean Local Training R2: {df_train_summary["train_r2"].mean():.4f}')
print(f'Mean Local Training MAE (log scale): {df_train_summary["train_mae_log"].mean():.4f}')

## 7. Prediction and Evaluation on Testing Data

Use the trained hierarchical model to predict views for the testing videos.

In [ ]:
testing_results = []
all_actual_log_views = []
all_predicted_log_views = []

current_test_idx = 0

for channel in test_data:
    channel_id = channel['channel_id']
    num_videos = len(channel['videos'])
    channel_test_pca = test_pca[current_test_idx : current_test_idx + num_videos, selected_dims]

    actual_views = [v['views'] for v in channel['videos']]
    actual_log_views = np.log1p(actual_views)

    model = channel_models[channel_id]
    predicted_norm_log_views = model.predict(channel_test_pca)
    predicted_log_views = predicted_norm_log_views * GLOBAL_STD + GLOBAL_MEAN

    test_r2 = r2_score(actual_log_views, predicted_log_views)
    test_mae = mean_absolute_error(actual_log_views, predicted_log_views)

    testing_results.append({
        'channel_name': channel['channel_name'],
        'num_test_videos': num_videos,
        'test_r2': test_r2,
        'test_mae': test_mae
    })

    all_actual_log_views.extend(actual_log_views)
    all_predicted_log_views.extend(predicted_log_views)

    current_test_idx += num_videos

df_test_summary = pd.DataFrame(testing_results)
print('Testing Summary (Top 10 Channels by R2):')
display(df_test_summary.sort_values('test_r2', ascending=False).head(10))

global_test_r2 = r2_score(all_actual_log_views, all_predicted_log_views)
global_test_mae = mean_absolute_error(all_actual_log_views, all_predicted_log_views)

print(f'\nAggregated Test R2 (all videos): {global_test_r2:.4f}')
print(f'Aggregated Test MAE (all videos, log scale): {global_test_mae:.4f}')
print(f'Mean Per-Channel Test R2: {df_test_summary["test_r2"].mean():.4f}')

## 8. Expanded Analysis and Debugging

Deeper analysis of the model results and feature importance.

In [ ]:
# 1. Global Feature Importance
plt.figure(figsize=(12, 6))
global_coeffs = global_model.params[1:]
global_pvalues = global_model.pvalues[1:]
importance_df = pd.DataFrame({
    'Dimension': [f'Dim_{i+1}' for i in range(CONFIG['GLOBAL_PCA_COMPONENTS'])],
    'Coefficient': global_coeffs,
    'P-Value': global_pvalues
})
importance_df = importance_df.sort_values('Coefficient', key=abs, ascending=False)

sns.barplot(x='Coefficient', y='Dimension', data=importance_df)
plt.title('Global Feature Importance (OLS Coefficients)')
plt.grid(True, axis='x')
plt.show()

# 2. Residual Analysis
all_actual_log_views = np.array(all_actual_log_views)
all_predicted_log_views = np.array(all_predicted_log_views)
residuals = all_actual_log_views - all_predicted_log_views

plt.figure(figsize=(10, 6))
plt.scatter(all_predicted_log_views, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Log Views')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(True)
plt.show()

# 3. Global vs. Local Weights
local_coeffs_list = [s['coeffs'] for s in training_summaries]
df_local_coeffs = pd.DataFrame(local_coeffs_list, columns=[f'Dim_{i+1}' for i in selected_dims])
avg_local_coeffs = df_local_coeffs.mean()

comp_df = pd.DataFrame({
    'Global Weight': [global_coeffs[i] for i in selected_dims],
    'Avg Local Weight': avg_local_coeffs.values
}, index=[f'Dim_{i+1}' for i in selected_dims])

comp_df.plot(kind='bar', figsize=(10, 6))
plt.title('Comparison: Global vs. Average Local Weights')
plt.ylabel('Coefficient Value')
plt.grid(True, axis='y')
plt.show()

# 4. Actual vs Predicted
plt.figure(figsize=(10, 8))
plt.scatter(all_actual_log_views, all_predicted_log_views, alpha=0.5, s=15)
plt.plot([all_actual_log_views.min(), all_actual_log_views.max()],
         [all_actual_log_views.min(), all_actual_log_views.max()], 'r--')
plt.xlabel('Actual Log Views')
plt.ylabel('Predicted Log Views')
plt.title('Actual vs Predicted Log Views (All Test Data)')
plt.grid(True)
plt.show()